# Minimalist QnA Bot (LangChain + Ollama)

Answers questions about ingredients, menu, and transactions using a local Ollama model.

In [1]:
import json
from pathlib import Path

from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

DATA_DIR = Path("../data")
MODEL = "qwen3.5:4b"

In [2]:
def load_context() -> str:
    files = ["ingredients.json", "menu.json", "transactions.json"]
    parts = []
    for f in files:
        data = json.loads((DATA_DIR / f).read_text())
        parts.append(f"## {f}\n{json.dumps(data)}")
    return "\n\n".join(parts)

context = load_context()
print(f"{len(context)} chars loaded")

4098 chars loaded


In [3]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer questions about a restaurant's stock inventory, menu, "
               "and transactions using only the data below. If the answer isn't "
               "in the data, say so.\n\n{context}"),
    ("human", "{question}"),
])

llm = ChatOllama(model=MODEL, temperature=0)
chain = prompt | llm

def ask(question: str) -> str:
    return chain.invoke({"context": context, "question": question}).content

In [4]:
print(ask("How much ground beef do we have in stock?"))

Based on the provided data, we have **25.5 kg** of **Ground Beef (80/20)** in stock.


In [5]:
while True:
    q = input("Ask (blank to quit): ").strip()
    if not q:
        break
    print(ask(q), "\n")